# Data Modeling Across a Company Lifecycle: The Case of ACME Corporation

## Project Overview

In this project, you will use skills you have learned to model data in relational, document, and graph databases in various stages of and for various use cases in ACME Corporation's lifecycle.

The project will include:
- Using PostgreSQL with sample datasets provided to create tables with specific schemas for:
  - Front-end applications in an OLTP use case (normalized)
  - OLAP use cases for analytics systems (Star Schema, Snowflake)
- Modeling data for a subsidiary of ACME Corporation in MongoDB for unstructured, flexible data
- Building a data model for a graph database using Neo4j to capture relationships between products, purchases, and customers

Once you complete this project, you will have demonstrated substantial skills in database normalization, denormalization, alongside relational and nonrelational data modeling. This project will test how well you understand the schemas we have covered, as well as the use cases. It also mirrors a practical example data engineers are often given in the workplace! You will have to brush up on your DDL/DML, SQL, MQL and Cypher skills. 

**Estimated Time:** 5 hours

## Technical Requirements

You are provided sidecar connections for the following tools and technologies in this workspace. These include:
- **PostgreSQL**
- **MongoDB**
- **Neo4j**

### Starter Files
Your workspace also contains sample datasets for:
- ACME Corporation: products, purchases, customers, and user ratings
- ACME 3D Printing: products, purchases, customers, and user ratings

**Note:** There are no costs associated with this project when using the classroom workspace.

## Project Scenario

You are working at **ACME Corporation**, an e-commerce company that sells two types of gears: large and small. The company's customers are a pre-defined set of wholesalers who place orders on the website. ACME uses a legacy infrastructure system that is only compatible with PostgreSQL.

---

# Project Steps

## Part 1: Relational Database Design with PostgreSQL

### Task 1.1: OLTP Database (Normalized Schema)

### Your Journey Through ACME's Growth

**Stage 1: Building the Foundation (Relational Databases)**  
You'll design a relational schema to handle transactional data for products, purchases, customers, and user ratings. You'll build two models:
- A normalized model for the transactional (OLTP) use case
- A star schema model for analytics (OLAP) use case

Create a **normalized set of tables** for the front-end application to handle transactional operations (OLTP). Your schema should include:
- Products
- Purchases (Orders)
- Customers
- User Ratings

**Requirements:**
- Normalize tables to at least 3rd Normal Form (3NF)
- Avoid all CRUD anomalies (insert, update, delete)
- Define appropriate primary keys and foreign keys
- Set proper constraints (NOT NULL, UNIQUE, CHECK, etc.)

**Deliverable:** Write DDL statements to create your normalized schema.

In [ ]:
# Setup: Import required libraries and establish PostgreSQL connection
import psycopg2
from psycopg2 import Error

def get_postgres_connection():
    """Create a PostgreSQL connection"""
    try:
        conn = psycopg2.connect(
            dbname='acme_db',
            user='postgres',
            password='your_password',  # Update with your password
            host='localhost',
            port='5432'
        )
        conn.autocommit = False
        print("✅ Connected to PostgreSQL")
        return conn
    except Error as e:
        print(f"❌ Error connecting to PostgreSQL: {e}")
        return None

In [ ]:
# Task 1.1: Create OLTP normalized schema
# TODO: Write your DDL statements here

oltp_schema = """
-- Drop existing tables if they exist
DROP TABLE IF EXISTS user_ratings CASCADE;
DROP TABLE IF EXISTS purchases CASCADE;
DROP TABLE IF EXISTS products CASCADE;
DROP TABLE IF EXISTS customers CASCADE;

-- Create Customers table
CREATE TABLE customers (
    customer_id SERIAL PRIMARY KEY,
    -- Add your columns here
);

-- Create Products table
CREATE TABLE products (
    product_id SERIAL PRIMARY KEY,
    -- Add your columns here
);

-- Create Purchases table
CREATE TABLE purchases (
    purchase_id SERIAL PRIMARY KEY,
    -- Add your columns here
    -- Add foreign keys to customers and products
);

-- Create User Ratings table
CREATE TABLE user_ratings (
    rating_id SERIAL PRIMARY KEY,
    -- Add your columns here
    -- Add foreign keys
);
"""

# Execute the schema
conn = get_postgres_connection()
if conn:
    cursor = conn.cursor()
    try:
        cursor.execute(oltp_schema)
        conn.commit()
        print("✅ OLTP schema created successfully")
    except Error as e:
        conn.rollback()
        print(f"❌ Error creating schema: {e}")
    finally:
        cursor.close()
        conn.close()

### Task 1.2: OLAP Database (Star Schema)

Create a **star schema** for analytics purposes (OLAP). This schema should be optimized for read performance rather than write performance.

**Requirements:**
- Design a fact table for purchases/transactions
- Create dimension tables for:
  - Products
  - Customers
  - Time (dates)
- Denormalize where appropriate for query performance
- Consider which metrics and dimensions analysts will need

**Deliverable:** Write DDL statements to create your star schema.

In [ ]:
# Task 1.2: Create OLAP star schema
# TODO: Write your DDL statements here

olap_schema = """
-- Drop existing tables if they exist
DROP TABLE IF EXISTS fact_purchases CASCADE;
DROP TABLE IF EXISTS dim_products CASCADE;
DROP TABLE IF EXISTS dim_customers CASCADE;
DROP TABLE IF EXISTS dim_time CASCADE;

-- Create Dimension: Time
CREATE TABLE dim_time (
    time_key SERIAL PRIMARY KEY,
    -- Add your columns here (date, day, month, quarter, year, etc.)
);

-- Create Dimension: Products
CREATE TABLE dim_products (
    product_key SERIAL PRIMARY KEY,
    -- Add your columns here (include descriptive attributes)
);

-- Create Dimension: Customers
CREATE TABLE dim_customers (
    customer_key SERIAL PRIMARY KEY,
    -- Add your columns here (include descriptive attributes)
);

-- Create Fact Table: Purchases
CREATE TABLE fact_purchases (
    purchase_key SERIAL PRIMARY KEY,
    time_key INT REFERENCES dim_time(time_key),
    product_key INT REFERENCES dim_products(product_key),
    customer_key INT REFERENCES dim_customers(customer_key),
    -- Add your measures here (quantities, amounts, etc.)
);
"""

# Execute the schema
conn = get_postgres_connection()
if conn:
    cursor = conn.cursor()
    try:
        cursor.execute(olap_schema)
        conn.commit()
        print("✅ OLAP star schema created successfully")
    except Error as e:
        conn.rollback()
        print(f"❌ Error creating schema: {e}")
    finally:
        cursor.close()
        conn.close()

---

## Part 2: NoSQL Database Design with MongoDB

### Task 2.1: Design MongoDB Collections for ACME 3D Printing

**Stage 2: Scaling Up (NoSQL Databases)**  
ACME Corporation is growing rapidly and launching a subsidiary: **ACME 3D Printing**. This startup helps customers create all kinds of custom hardware. Unlike the parent company, ACME 3D Printing moves fast and offers diverse services. The product catalog is constantly changing, making a fixed schema impractical. You'll use MongoDB to create a flexible data model.

ACME 3D Printing operates differently from its parent company. Products are constantly changing, each with different attributes. The e-commerce application must:

1. **Display the last 10 products or purchases** of each customer
2. **Display different products or purchases** in a catalog page, where each product has different attributes
3. **Provide each customer with information** about products used in their industry

**Data Modeling Requirements:**
- Include the last 10 products/purchases within the customer document (embed recent activity)
- Use flexible schemas where each product can have different attributes
- Embed industry-specific product information within customer documents
- Follow MongoDB best practices for document design

**Reference:** [MongoDB Data Modeling Guide](https://www.mongodb.com/docs/manual/data-modeling/)

**Deliverable:** Design MongoDB collections and sample documents.

In [ ]:
# Setup: Import MongoDB libraries
from pymongo import MongoClient
from datetime import datetime
import json

def get_mongo_connection():
    """Create a MongoDB connection"""
    try:
        client = MongoClient('mongodb://localhost:27017/')  # Update with your connection string
        db = client['acme_3d_printing']
        print("✅ Connected to MongoDB")
        return db
    except Exception as e:
        print(f"❌ Error connecting to MongoDB: {e}")
        return None

In [ ]:
# Task 2.1: Design Customer Collection with embedded data
# TODO: Create your document schema

# Example Customer Document Schema
customer_schema = {
    "_id": "ObjectId or custom ID",
    "customer_info": {
        "name": "string",
        "email": "string",
        "company": "string",
        "industry": "string"
    },
    # TODO: Add recent_purchases array (last 10 purchases embedded)
    "recent_purchases": [
        # Design your embedded purchase documents
    ],
    # TODO: Add industry_products array (relevant products for this customer's industry)
    "industry_products": [
        # Design your embedded product references
    ],
    "created_at": "ISODate",
    "updated_at": "ISODate"
}

print("Customer Collection Schema:")
print(json.dumps(customer_schema, indent=2))

In [ ]:
# Task 2.1: Design Product Collection with flexible schema
# TODO: Create your document schema

# Example Product Document Schema (flexible attributes)
product_schema = {
    "_id": "ObjectId or custom ID",
    "product_name": "string",
    "category": "string",
    "base_price": "decimal",
    # TODO: Add flexible attributes that vary by product type
    "attributes": {
        # Different products will have different attributes here
        # Example: color, material, dimensions, weight, etc.
    },
    "industries": ["array of industries this product serves"],
    "created_at": "ISODate",
    "updated_at": "ISODate"
}

print("Product Collection Schema:")
print(json.dumps(product_schema, indent=2))

In [ ]:
# Task 2.1: Create sample documents in MongoDB
# TODO: Insert sample documents

db = get_mongo_connection()
if db:
    # Example: Insert a sample customer
    sample_customer = {
        # TODO: Create a complete sample customer document
    }
    
    # Example: Insert a sample product
    sample_product = {
        # TODO: Create a complete sample product document
    }
    
    try:
        # db.customers.insert_one(sample_customer)
        # db.products.insert_one(sample_product)
        print("✅ Sample documents inserted")
    except Exception as e:
        print(f"❌ Error inserting documents: {e}")

### Task 2.2: Explain Your Design Decisions

Answer the following questions about your MongoDB design:

1. **Why did you choose to embed recent purchases in the customer document?**
2. **What are the trade-offs of embedding vs. referencing?**
3. **How does your schema handle products with different attributes?**
4. **What indexes would you create to optimize common queries?**

**Your Answer:**

1. TODO: Your explanation
2. TODO: Your explanation
3. TODO: Your explanation
4. TODO: Your explanation

---

## Part 3: Graph Database Design with Neo4j

### Task 3.1: Design Graph Schema for Recommendation Engine

**Stage 3: Intelligent Recommendations (Graph Databases)**  
ACME 3D Printing becomes so successful that it has thousands of products, overwhelming users. The product management team asks you to build a graph database schema in Neo4j to power a chatbot and recommendation engine.




ACME 3D Printing has thousands of products. The product management team needs a recommendation engine and chatbot powered by a graph database.

**Requirements:**
- Model relationships between:
  - Customers
  - Products
  - Purchases
  - Industries
  - Product Categories
- Design relationships that enable:
  - "Customers who bought X also bought Y"
  - "Products similar to X"
  - "Popular products in your industry"
  - "Customers similar to you"

**Deliverable:** Create nodes, relationships, and sample Cypher queries.

In [ ]:
# Setup: Import Neo4j libraries
from neo4j import GraphDatabase

class Neo4jConnection:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
    
    def close(self):
        self.driver.close()
    
    def execute_query(self, query, parameters=None):
        with self.driver.session() as session:
            result = session.run(query, parameters)
            return [record for record in result]

# Connect to Neo4j
# neo4j_conn = Neo4jConnection("bolt://localhost:7687", "neo4j", "your_password")
print("Neo4j connection setup ready")

### Task 3.1a: Define Node Types

Define the following node types and their properties:
- Customer
- Product
- Industry
- Category

In [ ]:
# Task 3.1a: Create nodes in Neo4j
# TODO: Write Cypher queries to create sample nodes

create_nodes_query = """
// Create Customer nodes
CREATE (c1:Customer {
    // TODO: Add customer properties
})

// Create Product nodes
CREATE (p1:Product {
    // TODO: Add product properties
})

// Create Industry nodes
CREATE (i1:Industry {
    // TODO: Add industry properties
})

// Create Category nodes
CREATE (cat1:Category {
    // TODO: Add category properties
})
"""

print("Node creation query:")
print(create_nodes_query)

### Task 3.1b: Define Relationships

Define the following relationship types:
- PURCHASED (Customer → Product)
- BELONGS_TO (Product → Category)
- SERVES (Product → Industry)
- WORKS_IN (Customer → Industry)
- SIMILAR_TO (Product → Product)
- ALSO_BOUGHT (Product → Product)

In [ ]:
# Task 3.1b: Create relationships in Neo4j
# TODO: Write Cypher queries to create relationships

create_relationships_query = """
// Create PURCHASED relationships
MATCH (c:Customer), (p:Product)
WHERE // TODO: Add your WHERE conditions
CREATE (c)-[:PURCHASED {
    // TODO: Add relationship properties (date, quantity, price, etc.)
}]->(p)

// Create BELONGS_TO relationships
MATCH (p:Product), (cat:Category)
WHERE // TODO: Add your WHERE conditions
CREATE (p)-[:BELONGS_TO]->(cat)

// TODO: Add more relationships
"""

print("Relationship creation query:")
print(create_relationships_query)

### Task 3.2: Write Recommendation Queries

Write Cypher queries for the following recommendation scenarios:

In [ ]:
# Task 3.2a: "Customers who bought X also bought Y"
# TODO: Write a Cypher query

collaborative_filtering_query = """
// Find products frequently purchased together
MATCH (p1:Product {name: 'Product X'})<-[:PURCHASED]-(c:Customer)-[:PURCHASED]->(p2:Product)
WHERE p1 <> p2
// TODO: Complete the query with aggregation and ordering
"""

print("Collaborative filtering query:")
print(collaborative_filtering_query)

In [ ]:
# Task 3.2b: "Popular products in your industry"
# TODO: Write a Cypher query

industry_popular_query = """
// Find popular products for a specific industry
MATCH (c:Customer {id: 'customer123'})-[:WORKS_IN]->(i:Industry)
// TODO: Complete the query to find popular products in that industry
"""

print("Industry popularity query:")
print(industry_popular_query)

In [ ]:
# Task 3.2c: "Products similar to X"
# TODO: Write a Cypher query

similar_products_query = """
// Find products similar to a given product
MATCH (p1:Product {name: 'Product X'})
// TODO: Complete the query using category, shared customers, or SIMILAR_TO relationships
"""

print("Similar products query:")
print(similar_products_query)

### Task 3.3: Explain Your Graph Design

Answer the following questions about your Neo4j design:

1. **What nodes and relationships did you create and why?**
2. **How does your graph structure enable personalized recommendations?**
3. **What properties did you add to relationships and why?**
4. **How would you handle graph growth as more customers and products are added?**

**Your Answer:**

1. TODO: Your explanation
2. TODO: Your explanation
3. TODO: Your explanation
4. TODO: Your explanation

---

## Project Reflection

### Summary of Your Work

Congratulations on completing all three parts of the project! You have:

1. ✅ Designed normalized and denormalized relational schemas in PostgreSQL
2. ✅ Created flexible document models in MongoDB
3. ✅ Built a graph database for recommendations in Neo4j

### Final Reflection Questions

1. **When would you choose a relational database vs. NoSQL vs. graph database?**
2. **What were the main trade-offs you encountered in each design?**
3. **How would you approach data migration between these different database types?**
4. **What additional features or optimizations would you add to these schemas in a production environment?**

**Your Reflection:**

1. TODO: Your answer
2. TODO: Your answer
3. TODO: Your answer
4. TODO: Your answer

---

## Standout Suggestion (Optional)

### AWS Deployment with CLI

Take your project to the next level by deploying all three databases on AWS:

- **PostgreSQL:** AWS RDS
- **MongoDB:** AWS DocumentDB or MongoDB Atlas
- **Graph Database:** AWS Neptune

Use the AWS CLI to:
1. Provision the database instances
2. Configure security groups and access
3. Execute your CREATE TABLE/COLLECTION/NODE statements
4. Load sample data
5. Test your queries

**Bonus:** Create Infrastructure as Code (IaC) using AWS CloudFormation or Terraform to automate the entire deployment.

In [ ]:
# Standout: AWS CLI commands (example)
# TODO: Document your AWS deployment process

aws_commands = """
# Create RDS PostgreSQL instance
aws rds create-db-instance \
    --db-instance-identifier acme-postgres \
    --db-instance-class db.t3.micro \
    --engine postgres \
    --master-username admin \
    --master-user-password YourPassword123 \
    --allocated-storage 20

# TODO: Add commands for DocumentDB and Neptune
"""

print("AWS deployment commands:")
print(aws_commands)